# Day 025 Project: AI Email-Responder Drafter

## What You're Building

An `EmailDrafter` class that takes an incoming email (raw RFC 2822 string), parses it, uses the LLM to draft a reply, and returns a fully formed `EmailMessage` ready to send via SMTP.

This is the pipeline: `parse → draft → build_reply → (optionally) send`.

## Project Requirements

1. Implement `EmailDrafter` with methods:
   - `parse(raw_email)` → `email.message.Message`
   - `draft(msg, context, model)` → `str` (reply body)
   - `build_reply(original_msg, reply_body, from_addr)` → `EmailMessage`
   - `run_pipeline(raw_email, context, from_addr, model)` → `dict`
2. Run the pipeline on SAMPLE_EMAIL with a context hint
3. Print the reply email headers and body

**Deliverable:** A complete reply email string — ready to paste into SMTP.

In [ ]:
import email
import email.message
import ollama

## Provided: All Helper Functions

In [ ]:
def build_email_message(
    to: str,
    subject: str,
    body: str,
    from_addr: str = "sender@example.com",
) -> email.message.EmailMessage:
    msg = email.message.EmailMessage()
    msg["From"] = from_addr
    msg["To"] = to
    msg["Subject"] = subject
    msg.set_content(body)
    return msg


def parse_email_string(raw_email: str) -> email.message.Message:
    return email.message_from_string(raw_email)


def get_email_body(msg: email.message.Message) -> str:
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                raw = part.get_payload(decode=True)
                if raw is not None:
                    charset = part.get_content_charset() or "utf-8"
                    return raw.decode(charset, errors="replace")
                return str(part.get_payload() or "")
    raw = msg.get_payload(decode=True)
    if raw is not None:
        charset = msg.get_content_charset() or "utf-8"
        return raw.decode(charset, errors="replace")
    return str(msg.get_payload() or "")


def draft_reply(
    original_subject: str,
    original_body: str,
    context: str,
    model: str = "llama3.2",
) -> str:
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a professional email assistant. "
                    "Write concise, polite email replies. "
                    "Match the tone of the original. "
                    "Return only the email body — no subject line, no 'Subject:' prefix."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Original subject: {original_subject}\n\n"
                    f"Original message:\n{original_body}\n\n"
                    f"Context for your reply: {context}\n\n"
                    "Write a reply email body:"
                ),
            },
        ],
    )
    return response["message"]["content"]


def build_reply_email(
    original_msg: email.message.Message,
    reply_body: str,
    from_addr: str,
    to_addr: str | None = None,
) -> email.message.EmailMessage:
    reply = email.message.EmailMessage()
    subject = original_msg.get("Subject", "")
    if not subject.startswith("Re:"):
        subject = f"Re: {subject}"
    reply["Subject"] = subject
    reply["From"] = from_addr
    reply["To"] = to_addr or original_msg.get("From", "")
    msg_id = original_msg.get("Message-ID")
    if msg_id:
        reply["In-Reply-To"] = msg_id
    reply.set_content(reply_body)
    return reply

## Your Implementation

Implement `EmailDrafter` by wiring together the helper functions.

In [ ]:
class EmailDrafter:
    def parse(self, raw_email: str) -> email.message.Message:
        # TODO: return parse_email_string(raw_email)
        pass

    def draft(
        self, msg: email.message.Message, context: str, model: str = 'llama3.2'
    ) -> str:
        # TODO: body = get_email_body(msg)
        # TODO: return draft_reply(
        #           original_subject=msg.get('Subject', ''),
        #           original_body=body, context=context, model=model)
        pass

    def build_reply(
        self, original_msg: email.message.Message,
        reply_body: str, from_addr: str
    ) -> email.message.EmailMessage:
        # TODO: return build_reply_email(original_msg, reply_body, from_addr=from_addr)
        pass

    def run_pipeline(
        self, raw_email: str, context: str,
        from_addr: str, model: str = 'llama3.2'
    ) -> dict:
        # TODO: parsed     = self.parse(raw_email)
        # TODO: reply_body = self.draft(parsed, context=context, model=model)
        # TODO: reply_msg  = self.build_reply(parsed, reply_body, from_addr=from_addr)
        # TODO: return {'parsed': parsed, 'reply_body': reply_body, 'reply_msg': reply_msg}
        pass

## Use Your Email Drafter

In [ ]:
SAMPLE_EMAIL = 'From: alice@example.com\nTo: bob@example.com\nSubject: Project Kickoff Meeting\nMessage-ID: <20260717090000.alice@example.com>\nDate: Mon, 17 Jul 2026 09:00:00 +0000\nContent-Type: text/plain; charset=utf-8\n\nHi Bob,\n\nCan we schedule a kickoff meeting for the new project?\nI am available Tuesday or Wednesday afternoon.\n\nBest,\nAlice\n'


In [ ]:
# drafter = EmailDrafter()
# result = drafter.run_pipeline(
#     raw_email=SAMPLE_EMAIL,
#     context='Accept the meeting, suggest Thursday at 2pm',
#     from_addr='bob@example.com',
# )
# print('Original:')
# print(f"  From: {result['parsed']['From']}")
# print(f"  Subject: {result['parsed']['Subject']}")
# print('\nDrafted reply:')
# print(result['reply_body'])
# print('\nReply email (ready to send):')
# print(result['reply_msg'].as_string())


## How to Actually Send (SMTP)

Once you have a reply `EmailMessage`, sending it is three lines:

```python
import smtplib

# Gmail example — needs an App Password (not your login password)
with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
    server.login('you@gmail.com', 'your-app-password')
    server.send_message(reply_msg)
```

For Outlook/Hotmail: `smtp.office365.com:587` with `SMTP` + `starttls()`.
For local testing: use `smtplib.SMTP('localhost', 1025)` with a [mailhog](https://github.com/mailhog/MailHog) test server.

**Credentials**: store in environment variables (Day 32), never hardcode.

## How to Read Email (IMAP)

```python
import imaplib, email

with imaplib.IMAP4_SSL('imap.gmail.com') as imap:
    imap.login('you@gmail.com', 'your-app-password')
    imap.select('INBOX')
    _, data = imap.search(None, 'UNSEEN')
    for msg_id in data[0].split()[-5:]:
        _, msg_data = imap.fetch(msg_id, '(RFC822)')
        raw = msg_data[0][1]  # bytes
        msg = email.message_from_bytes(raw)
        body = get_email_body(msg)
        reply = drafter.run_pipeline(msg.as_string(), 'Acknowledge receipt', 'me@example.com')
```


## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: EmailDrafter has all required methods
    try:
        assert 'EmailDrafter' in globals(), 'EmailDrafter not defined'
        for m in ('parse', 'draft', 'build_reply', 'run_pipeline'):
            assert hasattr(EmailDrafter, m), f'EmailDrafter missing: {m}'
        passed += 1; print('\u2705 Check 1: EmailDrafter has all required methods')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: drafter is an EmailDrafter
    try:
        assert 'drafter' in globals(), 'drafter not defined'
        assert isinstance(drafter, EmailDrafter), \
            f'drafter must be EmailDrafter, got {type(drafter)}'
        passed += 1; print('\u2705 Check 2: drafter is an EmailDrafter')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: result is a dict with required keys
    try:
        assert 'result' in globals(), 'result not defined'
        for key in ('parsed', 'reply_body', 'reply_msg'):
            assert key in result, f"result missing key '{key}': {list(result)}"
        passed += 1; print('\u2705 Check 3: result has parsed/reply_body/reply_msg')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: reply_body is a non-empty string
    try:
        assert 'result' in globals(), 'result not defined'
        rb = result.get('reply_body')
        assert isinstance(rb, str) and len(rb) > 10, \
            f'reply_body should be non-empty str, got {rb!r}'
        passed += 1; print(f'\u2705 Check 4: reply_body is {len(rb)} chars')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: reply_msg has Re: subject and correct From
    try:
        assert 'result' in globals(), 'result not defined'
        rm = result.get('reply_msg')
        assert rm is not None, 'reply_msg is None'
        assert rm['Subject'].startswith('Re:'), \
            f"reply Subject should start with 'Re:', got {rm['Subject']!r}"
        assert rm['From'] == 'bob@example.com', \
            f"reply From should be 'bob@example.com', got {rm['From']!r}"
        passed += 1; print(f'\u2705 Check 5: reply subject and From correct')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add a `classify_urgency(msg, model) -> str` method that uses the LLM to classify the email as 'urgent', 'normal', or 'low-priority' before drafting
- Add a `summarize(msg, model) -> str` method that summarises the email in one sentence (useful when the body is long)
- Extend the pipeline to handle a list of raw emails: process each one, skipping emails where `get_email_body` returns an empty string
- Test with a real IMAP connection to your Gmail or Outlook (needs app password)
- Add `References` header alongside `In-Reply-To` for proper thread linking